In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# ================================
# KONFIGURASI 
# ================================
CONFIGS = [
    {
        "nama":  "Kata",
        "file":  "data_kata2.csv",
        "model": "model_kata.pkl",
        "mode":  "kata",          # pakai header + drop kolom "label
    },
    {
        "nama":  "Huruf (Alfabet)",
        "file":  "alfabet.csv",
        "model": "model_alfabet.pkl",
        "mode":  "huruf_angka",   # tanpa header, kolom terakhir = label
    },
    {
        "nama":  "Angka",
        "file":  "angka3.csv",
        "model": "model_angka.pkl",
        "mode":  "huruf_angka",
    },
]

# ================================
# FUNGSI LOAD DATA
# ================================
def load_data(cfg):
    if cfg["mode"] == "kata":
        df = pd.read_csv(cfg["file"])
        X = df.drop("label", axis=1).values
        y = df["label"].values
    else:  # huruf_angka
        df = pd.read_csv(cfg["file"], header=None)
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
    return X, y

# ================================
# FUNGSI TRAINING
# ================================
def train(cfg, X, y):
    if cfg["mode"] == "kata":
        # Optimized RF untuk kata
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        stratify = y
    else:
        # RF standar untuk huruf/angka
        model = RandomForestClassifier(n_estimators=200)
        stratify = None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify
    )
    model.fit(X_train, y_train)
    return model, X_test, y_test

# ================================
# MAIN LOOP
# ================================
for cfg in CONFIGS:
    print(f"\n{'='*50}")
    print(f"  Training: {cfg['nama']}")
    print(f"{'='*50}")

    # Load
    X, y = load_data(cfg)
    print(f"  Total data  : {len(X)}")
    print(f"  Jumlah kelas: {len(set(y))}")
    print(f"  Jumlah fitur: {X.shape[1]}")

    # Train
    model, X_test, y_test = train(cfg, X, y) #melatih model

    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n  Akurasi: {acc * 100:.2f}%")

    if cfg["mode"] == "kata":
        print("\n  Classification Report:")
        print(classification_report(y_test, y_pred))

    # Simpan
    joblib.dump(model, cfg["model"])
    print(f"  Model tersimpan: {cfg['model']}")

print("\nSemua model selesai ditraining!")


  Training: Kata
  Total data  : 1190
  Jumlah kelas: 7
  Jumlah fitur: 114

  Akurasi: 94.12%

  Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.97      0.90        34
           1       1.00      0.94      0.97        33
           2       0.94      0.91      0.93        34
           3       0.91      0.91      0.91        34
           4       1.00      1.00      1.00        33
           5       0.91      0.89      0.90        35
           6       1.00      0.97      0.99        35

    accuracy                           0.94       238
   macro avg       0.94      0.94      0.94       238
weighted avg       0.94      0.94      0.94       238

  Model tersimpan: model_kata.pkl

  Training: Huruf (Alfabet)
  Total data  : 21357
  Jumlah kelas: 29
  Jumlah fitur: 42

  Akurasi: 96.54%
  Model tersimpan: model_alfabet.pkl

  Training: Angka
  Total data  : 3800
  Jumlah kelas: 10
  Jumlah fitur: 46

  Akurasi: 99.47%
  Mod